**Start with cleaining audio files. All files are to be .WAV. All files are to be centered, cut to 3 seconds, and padded if too short.**

In [1]:
import os
from pathlib import Path
import librosa
import soundfile as sf
import numpy as np

REAL_INPUT_DIR = "data/realAudio"
FAKE_INPUT_DIR = "data/fakeAudio"

CLEAN_REAL_DIR = "data/cleanReal"
CLEAN_FAKE_DIR = "data/cleanFake"

TARGET_SR = 16000
TARGET_DURATION = 3.0 #in seconds


def standardizeAudio(in_path, out_path_base, sr=TARGET_SR, duration=TARGET_DURATION):
    try:
        y, _ = librosa.load(in_path, sr=sr, mono=True)
    except Exception as e:
        print(f"Could not load {in_path}: {e}")
        return

    target_len = int(sr * duration)

    # If too short, pad
    if len(y) < target_len:
        pad_total = target_len - len(y)
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        y = np.pad(y, (pad_left, pad_right))
    else:
        # Extract middle segment
        mid = len(y) // 2
        half = target_len // 2
        start = max(0, mid - half)
        y = y[start:start + target_len]

    # save as WAV
    out_path = Path(out_path_base).with_suffix(".wav")

    # Ensure directory exists
    os.makedirs(out_path.parent, exist_ok=True)

    # Save WAV
    sf.write(out_path, y, sr)
    print(f"Saved standardized WAV: {out_path}")


def processFolder(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    audio_files = list(input_dir.iterdir())

    if not audio_files:
        print(f"No audio files found in {input_dir}")
        return
    for f in audio_files:
        out_name = f.stem + "_clean"
        out_path = output_dir / out_name
        standardizeAudio(str(f), str(out_path))

def main():
    print("Processing REAL audio…")
    processFolder(REAL_INPUT_DIR, CLEAN_REAL_DIR)
    print("\nProcessing FAKE audio…")
    processFolder(FAKE_INPUT_DIR, CLEAN_FAKE_DIR)


if __name__ == "__main__":
    main()

Processing REAL audio…
Saved standardized WAV: data/cleanReal/mckinley2_clean.wav
Saved standardized WAV: data/cleanReal/mckinley3_clean.wav
Saved standardized WAV: data/cleanReal/mckinley1_clean.wav
Saved standardized WAV: data/cleanReal/zach8_clean.wav
Saved standardized WAV: data/cleanReal/mckinley4_clean.wav
Saved standardized WAV: data/cleanReal/mckinley10_clean.wav
Saved standardized WAV: data/cleanReal/mckinley5_clean.wav
Saved standardized WAV: data/cleanReal/zach9_clean.wav
Saved standardized WAV: data/cleanReal/mckinley7_clean.wav
Saved standardized WAV: data/cleanReal/mckinley6_clean.wav
Saved standardized WAV: data/cleanReal/zach15_clean.wav
Saved standardized WAV: data/cleanReal/mckinley11_clean.wav
Saved standardized WAV: data/cleanReal/zach14_clean.wav
Saved standardized WAV: data/cleanReal/mckinley12_clean.wav
Saved standardized WAV: data/cleanReal/mckinley13_clean.wav
Saved standardized WAV: data/cleanReal/zach13_clean.wav
Saved standardized WAV: data/cleanReal/zach12_

/var/folders/l1/991h4ps92ps2j8xfn3ycczzm0000gn/T/ipykernel_66422/1805184187.py:19: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(in_path, sr=sr, mono=True)
/opt/anaconda3/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [2]:
import os
from pathlib import Path

import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

REAL_INPUT_DIR = "data/cleanReal"
FAKE_INPUT_DIR = "data/cleanFake"

REAL_OUTPUT_DIR = "data/realSpect"
FAKE_OUTPUT_DIR = "data/fakeSpect"

SAMPLE_RATE = 16000


def create_spectrogram(wav_path, png_path, sr=SAMPLE_RATE):
    # Load audio
    y, sr = librosa.load(wav_path, sr=sr, mono=True)
    # Mel spectrogram
    S = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=2048,
        hop_length=512,
        n_mels=128
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    # Plot
    plt.figure(figsize=(8, 3))
    librosa.display.specshow(
        S_db, sr=sr, hop_length=512,
        x_axis="time", y_axis="mel"
    )
    plt.colorbar(label="dB")
    plt.title(Path(wav_path).name)
    plt.tight_layout()
    # Save png
    os.makedirs(os.path.dirname(png_path), exist_ok=True)
    plt.savefig(png_path, dpi=150)
    plt.close()

    print(f"Saved: {png_path}")


def processFolder(input_dir, output_dir):
    # Create spectrogram PNGs for all WAV files in a folder
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    wav_files = sorted(input_dir.glob("*.wav"))
    if not wav_files:
        print(f"No .wav files found in {input_dir}")
        return

    for wav_file in wav_files:
        out_name = wav_file.stem + "_spect.png"
        out_path = output_dir / out_name
        create_spectrogram(str(wav_file), str(out_path))


def main():
    print("Processing cleaned REAL audio")
    processFolder(REAL_INPUT_DIR, REAL_OUTPUT_DIR)

    print("\nProcessing cleaned FAKE audio")
    processFolder(FAKE_INPUT_DIR, FAKE_OUTPUT_DIR)


if __name__ == "__main__":
    main()

Processing cleaned REAL audio
Saved: data/realSpect/mckinley10_clean_spect.png
Saved: data/realSpect/mckinley11_clean_spect.png
Saved: data/realSpect/mckinley12_clean_spect.png
Saved: data/realSpect/mckinley13_clean_spect.png
Saved: data/realSpect/mckinley14_clean_spect.png
Saved: data/realSpect/mckinley15_clean_spect.png
Saved: data/realSpect/mckinley1_clean_spect.png
Saved: data/realSpect/mckinley2_clean_spect.png
Saved: data/realSpect/mckinley3_clean_spect.png
Saved: data/realSpect/mckinley4_clean_spect.png
Saved: data/realSpect/mckinley5_clean_spect.png
Saved: data/realSpect/mckinley6_clean_spect.png
Saved: data/realSpect/mckinley7_clean_spect.png
Saved: data/realSpect/mckinley8_clean_spect.png
Saved: data/realSpect/mckinley9_clean_spect.png
Saved: data/realSpect/zach10_clean_spect.png
Saved: data/realSpect/zach11_clean_spect.png
Saved: data/realSpect/zach12_clean_spect.png
Saved: data/realSpect/zach13_clean_spect.png
Saved: data/realSpect/zach14_clean_spect.png
Saved: data/realSpe

In [3]:
import os
import glob
import numpy as np
import pandas as pd
import librosa

def extractFingerprint(filepath, sr=16000, n_mfcc=13):
    # Load audio 
    y, sr = librosa.load(filepath, sr=sr, mono=True)
    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc,
        dct_type=2, norm='ortho'   # for stable reproducibility
    )

    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = float(mfcc[i].mean())
        features[f"mfcc_{i+1}_std"] = float(mfcc[i].std())

    # Spectral features
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.95)
    flatness = librosa.feature.spectral_flatness(y=y)

    features["spect_centroid_mean"] = float(centroid.mean())
    features["spect_centroid_std"] = float(centroid.std())

    features["spect_bandwidth_mean"] = float(bandwidth.mean())
    features["spect_bandwidth_std"] = float(bandwidth.std())

    features["spect_rolloff95_mean"] = float(rolloff.mean())
    features["spect_rolloff95_std"] = float(rolloff.std())

    features["spect_flatness_mean"] = float(flatness.mean())
    features["spect_flatness_std"] = float(flatness.std())

    # Temporal / energy
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    flux = librosa.onset.onset_strength(y=y, sr=sr)

    features["zcr_mean"] = float(zcr.mean())
    features["zcr_std"] = float(zcr.std())

    features["rms_mean"] = float(rms.mean())
    features["rms_std"] = float(rms.std())

    features["spec_flux_mean"] = float(flux.mean())
    features["spec_flux_std"] = float(flux.std())

    return features


def processFolders(real_dir, fake_dir):
    rows = []
    # Real
    for filepath in glob.glob(os.path.join(real_dir, "*.wav")):
        print(f"Processing (real): {filepath}")
        feats = extractFingerprint(filepath)
        feats["label"] = "real"
        feats["file"] = os.path.basename(filepath)
        rows.append(feats)

    # Fake
    for filepath in glob.glob(os.path.join(fake_dir, "*.wav")):
        print(f"Processing (deepfake): {filepath}")
        feats = extractFingerprint(filepath)
        feats["label"] = "deepfake"
        feats["file"] = os.path.basename(filepath)
        rows.append(feats)

    return pd.DataFrame(rows)


if __name__ == "__main__":
    REAL_DIR = "data/cleanReal"
    FAKE_DIR = "data/cleanFake"
    OUTPUT_CSV = "data/spectralFingerprints.csv"

    df = processFolders(REAL_DIR, FAKE_DIR)
    print("Extracted features shape:", df.shape)
    print(df.head())

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved features to {OUTPUT_CSV}")

Processing (real): data/cleanReal/zach1_clean.wav
Processing (real): data/cleanReal/zach15_clean.wav
Processing (real): data/cleanReal/mckinley7_clean.wav
Processing (real): data/cleanReal/zach13_clean.wav
Processing (real): data/cleanReal/mckinley1_clean.wav
Processing (real): data/cleanReal/zach7_clean.wav
Processing (real): data/cleanReal/mckinley11_clean.wav
Processing (real): data/cleanReal/mckinley6_clean.wav
Processing (real): data/cleanReal/zach14_clean.wav
Processing (real): data/cleanReal/zach12_clean.wav
Processing (real): data/cleanReal/zach6_clean.wav
Processing (real): data/cleanReal/mckinley10_clean.wav
Processing (real): data/cleanReal/mckinley5_clean.wav
Processing (real): data/cleanReal/mckinley15_clean.wav
Processing (real): data/cleanReal/zach3_clean.wav
Processing (real): data/cleanReal/mckinley8_clean.wav
Processing (real): data/cleanReal/zach5_clean.wav
Processing (real): data/cleanReal/mckinley13_clean.wav
Processing (real): data/cleanReal/zach11_clean.wav
Proce

**This is the to determine the suitable number of PCA components.**

In [20]:
'''import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

def find_optimal_pca(csv_path=CSV_PATH, variance_threshold=0.95):
    # Load data
    df = pd.read_csv(csv_path)

    # Separate features
    X = df.drop(columns=["label", "file"])

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA with all components
    pca_full = PCA()
    pca_full.fit(X_scaled)

    # Explained variance
    explained = pca_full.explained_variance_ratio_
    cumulative = explained.cumsum()

    # Find smallest k such that cumulative variance >= threshold
    k_opt = np.argmax(cumulative >= variance_threshold) + 1

    print("\nExplained variance ratio (per component):")
    print(explained)
    print("\nCumulative explained variance:")
    print(cumulative)
    print(f"\nOptimal number of components for {variance_threshold} variance: {k_opt}")

    return k_opt, cumulative


if __name__ == "__main__":
    k_opt, cumulative = find_optimal_pca()'''


Explained variance ratio (per component):
[2.73140233e-01 2.09817150e-01 1.18417621e-01 5.88293444e-02
 5.07226332e-02 4.62460680e-02 3.70127761e-02 3.18161225e-02
 2.74364823e-02 2.17518715e-02 1.94419054e-02 1.71522251e-02
 1.42020027e-02 1.06546532e-02 9.58342544e-03 8.48530103e-03
 7.60559909e-03 5.96531276e-03 5.54796206e-03 4.90302388e-03
 4.31939669e-03 3.53125223e-03 2.50303766e-03 2.16771690e-03
 1.79726374e-03 1.64652660e-03 1.36742418e-03 9.23094074e-04
 8.79568615e-04 5.22464452e-04 4.11370420e-04 3.10242244e-04
 2.86760672e-04 1.63166570e-04 1.60492951e-04 1.20970289e-04
 7.14210696e-05 3.95142678e-05 2.99501159e-05 1.66538129e-05]

Cumulative explained variance:
[0.27314023 0.48295738 0.601375   0.66020435 0.71092698 0.75717305
 0.79418583 0.82600195 0.85343843 0.8751903  0.89463221 0.91178443
 0.92598643 0.93664109 0.94622451 0.95470981 0.96231541 0.96828073
 0.97382869 0.97873171 0.98305111 0.98658236 0.9890854  0.99125312
 0.99305038 0.99469691 0.99606433 0.99698742 0

**This is the PCA algorithm and it's effect on the code. Accuracy with PCA + KNN = 0.5 (decrease from 0.83).**

In [22]:
'''import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

def run_pca(csv_path=CSV_PATH, n_components=17, top_features=5):
    # Load dataset
    df = pd.read_csv(csv_path)

    # Separate features and labels
    X = df.drop(columns=["label", "file"])
    y = df["label"]
    file_names = df["file"]

    feature_names = X.columns.tolist()

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Display explained variance
    print("\nExplained Variance Ratio")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"PC{i+1}: {var:.4f}")

    print("\nTOP CONTRIBUTING FEATURES PER PCA COMPONENT")
    loadings = pca.components_ 

    for i in range(n_components):
        component = loadings[i]
        # Sort features by absolute loading strength
        sorted_idx = np.argsort(np.abs(component))[::-1]
        top_idx = sorted_idx[:top_features]

        print(f"\nTop {top_features} features contributing to PC{i+1}:")
        for idx in top_idx:
            print(f"{feature_names[idx]} (loading={component[idx]:.4f})")

    # Build final PCA dataset
    df_final = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n_components)])
    df_final["label"] = y
    df_final["file"] = file_names

    return df_final, pca


if __name__ == "__main__":
    df_final, pca_model = run_pca()
    print("\nPCA-Reduced DataFrame:")
    print(df_final.head())'''


Explained Variance Ratio
PC1: 0.2731
PC2: 0.2098
PC3: 0.1184
PC4: 0.0588
PC5: 0.0507
PC6: 0.0462
PC7: 0.0370
PC8: 0.0318
PC9: 0.0274
PC10: 0.0218
PC11: 0.0194
PC12: 0.0172
PC13: 0.0142
PC14: 0.0107
PC15: 0.0096
PC16: 0.0085
PC17: 0.0076

TOP CONTRIBUTING FEATURES PER PCA COMPONENT

Top 5 features contributing to PC1:
zcr_mean (loading=0.2584)
spect_centroid_mean (loading=0.2552)
mfcc_2_std (loading=0.2510)
rms_mean (loading=0.2429)
mfcc_2_mean (loading=-0.2418)

Top 5 features contributing to PC2:
spec_flux_mean (loading=0.2869)
spect_flatness_mean (loading=-0.2827)
spect_bandwidth_std (loading=-0.2788)
spect_flatness_std (loading=-0.2672)
spect_rolloff95_std (loading=-0.2655)

Top 5 features contributing to PC3:
mfcc_3_mean (loading=0.3131)
spect_bandwidth_mean (loading=0.3079)
mfcc_6_mean (loading=0.2674)
spect_rolloff95_mean (loading=0.2658)
spect_centroid_std (loading=-0.2474)

Top 5 features contributing to PC4:
mfcc_10_mean (loading=0.3050)
mfcc_4_std (loading=-0.3041)
mfcc_7_me

**This is the OPTICS algorithm and it's effect on the code. Accuracy with OPTICS + KNN = 0.71 (decrease from 0.83).**

In [4]:
'''from sklearn.cluster import OPTICS
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

# Load data
df = pd.read_csv(CSV_PATH)

# Separate features and non-features
X = df.drop(columns=["label", "file"])
y = df["label"]

# SCALE FEATURES
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run OPTICS on scaled data
optics = OPTICS(
    max_eps=8,
    min_samples=2,
    metric='euclidean',
    min_cluster_size=2
)

optics.fit(X_scaled)

labels = optics.labels_
df["cluster"] = labels

# Check if any real clusters exist
non_noise_labels = [lab for lab in labels if lab != -1]

if len(non_noise_labels) == 0:
    print("\n[WARNING] OPTICS FOUND NO CLUSTERS — ALL POINTS LABELED -1")
    print("OPTICS is NOT removing outliers. Returning full dataset unchanged.")
    # Use full df as final result (just drop cluster column)
    df_final = df.drop(columns=["cluster"])

else:
    # Identify outliers
    indices_to_drop = df[df["cluster"] == -1].index
    print("\nOUTLIER INDICES TO DROP (-1 labels):")
    print(indices_to_drop.tolist())

    print(f"\nOutliers detected: {len(indices_to_drop)}")
    print(f"Total samples: {len(df)}")
    print(f"Remaining samples: {len(df) - len(indices_to_drop)}")

    # Drop outliers
    df_new = df.drop(indices_to_drop).reset_index(drop=True)

    # Final cleaned dataset
    df_final = df_new.drop(columns=["cluster"])'''


OUTLIER INDICES TO DROP (-1 labels):
[0, 12, 19, 23, 25, 32, 36, 37, 41, 42, 47, 49, 51, 52, 56, 57]

Outliers detected: 16
Total samples: 60
Remaining samples: 44


**This is the isolation forest that gives us the outliers. By removing said outliers, we were able to get an improved accuracy of 0.87 (compared to 0.83)**

In [4]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"
df = pd.read_csv(CSV_PATH)

# Separate features/labels
X = df.drop(columns=["label", "file"])
y = df["label"]

clf = IsolationForest( random_state=0)

outlier_flags = clf.fit_predict(X)   # -1 = outlier, 1 = inlier
df['outlier'] = outlier_flags

# Summary counts\
n_outliers = np.sum(outlier_flags == -1)
n_inliers = np.sum(outlier_flags == 1)

print("OUTLIER STATISTICS")
print(f"Total outliers detected: {n_outliers}")
print(f"Total inliers: {n_inliers}")

# Show which indices will be dropped
indices_to_drop = df[df['outlier'] == -1].index
print("OUTLIER INDICES")
print(indices_to_drop.tolist())

# Drop outliers
df_new = df[df['outlier'] == 1].reset_index(drop=True)

# Final cleaned dataset
df_final = df_new.drop(columns=["outlier"])

OUTLIER STATISTICS
Total outliers detected: 10
Total inliers: 50
OUTLIER INDICES
[2, 10, 13, 14, 25, 30, 35, 46, 50, 55]


**KNN training. If you wish to run this on the orginal data, change 'df_final' to 'df'.**

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneOut

# Path to your CSV from the feature-extraction script
CSV_PATH = "data/spectralFingerprints.csv"

def main():
    # Load data
    df = pd.read_csv(CSV_PATH)

    
    # Drop non-feature columns
    X = df_final.drop(columns=["label", "file"])
    y = df_final["label"]
    # X = df.drop(columns=["label", "file"])
    # y = df["label"]

    loo = LeaveOneOut()
    print(loo.get_n_splits(X))
    avg_result = 0
    for i, (train_index, test_index) in enumerate(loo.split(X)):
        #print(f"Fold {i}:")
        #print(f"  Train: index={train_index}")
        #print(f"  Test:  index={test_index}")

        X_train = X.iloc[train_index]
        y_train = y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test = y.iloc[test_index]

        clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
        ])

        # Train
        clf.fit(X_train, y_train)

        # Evaluate
        acc = clf.score(X_test, y_test)
        #print(f"Accuracy: {acc:.4f}\n")
        avg_result += acc
    print('Avg. Accuracy with every datapoint as the test sample:')
    print(avg_result/loo.get_n_splits(X))

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Pipeline: standardize features -> KNN
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ])

    # Train
    clf.fit(X_train, y_train)

    print("\nCross validation score:")
    print(cross_val_score(clf, X_train, y_train))
    print()

    # Evaluate
    acc = clf.score(X_test, y_test)
    print(f"Accuracy: {acc:.4f}\n")

    y_pred = clf.predict(X_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    main()

50
Avg. Accuracy with every datapoint as the test sample:
0.8

Cross validation score:
[0.57142857 0.85714286 0.85714286 1.         0.85714286]

Accuracy: 0.8667

Confusion Matrix:
[[6 2]
 [0 7]]

Classification Report:
              precision    recall  f1-score   support

    deepfake       1.00      0.75      0.86         8
        real       0.78      1.00      0.88         7

    accuracy                           0.87        15
   macro avg       0.89      0.88      0.87        15
weighted avg       0.90      0.87      0.87        15

